# 1-Step vs h-Step Ahead Forecasts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/EMQA/blob/main/EMQA_forecast_horizon/EMQA_forecast_horizon.ipynb)

## Overview

This quantlet demonstrates the difference between **1-step ahead rolling forecasts** and **h-step ahead forecasts** for AR(1) processes.

### Key Concepts

**1-Step Rolling Forecast:**
- Forecast one period ahead, observe actual, then forecast again
- Uses latest information at each step
- Constant forecast standard error: $\sigma_1 = \sigma$
- More accurate but requires continuous updates

**h-Step Ahead Forecast:**
- Forecast multiple periods from single point in time
- For AR(1): $\hat{y}_{T+h} = \mu + \phi^h(y_T - \mu)$
- Forecast converges to mean as $h \to \infty$
- Standard error grows: $\sigma_h = \sigma\sqrt{\frac{1-\phi^{2h}}{1-\phi^2}}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Chart styling
plt.rcParams.update({
    'figure.facecolor': 'none',
    'axes.facecolor': 'none',
    'savefig.facecolor': 'none',
    'font.size': 11
})

BLUE = '#1f77b4'
RED = '#d62728'
GREEN = '#2ca02c'
GRAY = '#7f7f7f'

## 1. Simulate AR(1) Process

In [ ]:
np.random.seed(42)

# AR(1) parameters
phi = 0.85    # persistence (try 0.5, 0.85, 0.95)
mu = 0        # long-run mean
sigma = 1     # innovation std
n_obs = 50    # historical observations
n_forecast = 20  # forecast horizon

# Simulate AR(1): y_t = mu + phi*(y_{t-1} - mu) + epsilon_t
y = np.zeros(n_obs + n_forecast)
y[0] = 0
for t in range(1, len(y)):
    y[t] = mu + phi * (y[t-1] - mu) + np.random.normal(0, sigma)

y_history = y[:n_obs]
y_future = y[n_obs:]

print(f"AR(1) with φ = {phi}")
print(f"Half-life of shocks: {np.log(0.5)/np.log(phi):.1f} periods")
print(f"Last observed value: y_T = {y_history[-1]:.2f}")

## 2. Generate Forecasts

In [ ]:
y_T = y_history[-1]  # Last observation
h_steps = np.arange(1, n_forecast + 1)

# h-step ahead forecasts (from single point T)
# Formula: E[y_{T+h}|y_T] = mu + phi^h * (y_T - mu)
h_step_forecasts = mu + (phi ** h_steps) * (y_T - mu)

# h-step forecast standard errors
# Var(y_{T+h}) = sigma^2 * (1 - phi^{2h}) / (1 - phi^2)
h_step_se = sigma * np.sqrt((1 - phi**(2*h_steps)) / (1 - phi**2))

# 1-step rolling forecasts (use actual as it becomes available)
one_step_forecasts = np.zeros(n_forecast)
for h in range(n_forecast):
    if h == 0:
        one_step_forecasts[h] = mu + phi * (y_T - mu)
    else:
        one_step_forecasts[h] = mu + phi * (y_future[h-1] - mu)

one_step_se = sigma  # constant for 1-step

print("\nh-step forecasts converge to mean:")
print(f"  h=1:  {h_step_forecasts[0]:.2f} (SE={h_step_se[0]:.2f})")
print(f"  h=5:  {h_step_forecasts[4]:.2f} (SE={h_step_se[4]:.2f})")
print(f"  h=20: {h_step_forecasts[19]:.2f} (SE={h_step_se[19]:.2f})")
print(f"\n1-step SE is constant: {one_step_se:.2f}")

## 3. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_alpha(0)

# --- Left panel: Time series ---
ax1 = axes[0]
ax1.patch.set_alpha(0)

time_history = np.arange(n_obs)
time_future = np.arange(n_obs, n_obs + n_forecast)

ax1.plot(time_history, y_history, color=BLUE, lw=1.5, label='Historical')
ax1.plot(time_future, y_future, color='black', lw=2, marker='o', ms=4, label='Actual')
ax1.plot(time_future, h_step_forecasts, color=RED, lw=2, ls='--', label='h-step forecast')
ax1.fill_between(time_future, h_step_forecasts - 1.96*h_step_se,
                  h_step_forecasts + 1.96*h_step_se, color=RED, alpha=0.15)
ax1.plot(time_future, one_step_forecasts, color=GREEN, lw=2, ls=':', label='1-step rolling')
ax1.fill_between(time_future, one_step_forecasts - 1.96*one_step_se,
                  one_step_forecasts + 1.96*one_step_se, color=GREEN, alpha=0.15)
ax1.axhline(y=mu, color=GRAY, ls='--', alpha=0.5)
ax1.axvline(x=n_obs-0.5, color=GRAY, ls=':', alpha=0.7)

ax1.set_xlabel('Time', fontweight='bold')
ax1.set_ylabel('Value', fontweight='bold')
ax1.set_title(f'AR(1) Forecasting (φ = {phi})', fontweight='bold')
ax1.set_xlim(30, n_obs + n_forecast)
ax1.grid(False)

# --- Right panel: Errors ---
ax2 = axes[1]
ax2.patch.set_alpha(0)

h_step_errors = np.abs(y_future - h_step_forecasts)
one_step_errors = np.abs(y_future - one_step_forecasts)

show_n = 10
width = 0.35
x = np.arange(1, show_n + 1)

ax2.bar(x - width/2, h_step_errors[:show_n], width, color=RED, alpha=0.7, label='h-step error')
ax2.bar(x + width/2, one_step_errors[:show_n], width, color=GREEN, alpha=0.7, label='1-step error')
ax2.plot(x, h_step_se[:show_n], color=RED, lw=2, ls='--', label='h-step SE')
ax2.axhline(y=one_step_se, color=GREEN, lw=2, ls=':', label='1-step SE')

ax2.set_xlabel('Forecast Horizon (h)', fontweight='bold')
ax2.set_ylabel('Absolute Error', fontweight='bold')
ax2.set_title('Forecast Error by Horizon', fontweight='bold')
ax2.set_xticks(x)
ax2.grid(False)

# Legend outside bottom
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
fig.legend(handles1 + handles2, labels1 + labels2,
           loc='lower center', bbox_to_anchor=(0.5, -0.02),
           ncol=4, frameon=False, fontsize=9)

plt.tight_layout()
plt.subplots_adjust(bottom=0.18)
plt.savefig('forecast_horizon_comparison.pdf', bbox_inches='tight', dpi=300, facecolor='none', edgecolor='none')
plt.savefig('forecast_horizon_comparison.png', bbox_inches='tight', dpi=300, facecolor='none', edgecolor='none')
plt.show()

## 4. Key Takeaways

| Aspect | 1-Step Rolling | h-Step Ahead |
|--------|---------------|---------------|
| Information | Uses latest actual | Single forecast point |
| Accuracy | Higher (constant SE) | Lower (SE grows with h) |
| Forecast path | Tracks actual | Converges to μ |
| Use case | Real-time trading | Budget planning |

### Formulas

**AR(1) h-step forecast:**
$$\hat{y}_{T+h|T} = \mu + \phi^h(y_T - \mu)$$

**h-step standard error:**
$$\sigma_h = \sigma\sqrt{\frac{1-\phi^{2h}}{1-\phi^2}}$$

As $h \to \infty$: forecast → $\mu$, SE → $\sigma/\sqrt{1-\phi^2}$ (unconditional std)